# Evaluation Pipeline

End-to-end notebook for running the 4-stage evaluation pipeline and analysing results.  
Each stage is a **config cell → launch cell → progress cell** triple.

| Stage | What it does | Output |
|-------|-------------|--------|
| 1 — Worlds | Generate synthetic travel worlds | `worlds/{SET_NAME}/world_*/` |
| 2 — Requests | Generate user requests bound to worlds | `data/user_requests/{COLLECTION}/request_*.json` |
| 3 — Agent Run | Run ReAct agent to produce episode logs | `outputs/episodes/ep_*.json` |
| 4 — Score | Evaluate episodes (deterministic ± LLM judge) | `outputs/eval_results/{run_id}/` |

**Progress checking:** re-run the `[Progress]` cell for any stage — it counts output files on disk.  
All subprocesses are non-blocking (Popen). You can run later cells while an earlier stage is still running.

**Analysis utilities** are imported from `evaluation/results_loader.py`.

## 0 · Setup

In [ ]:
import os
import subprocess
import sys
import time
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd

# ── Repo root resolution (works whether run from notebooks/ or repo root) ──────
REPO_ROOT = Path(globals().get('__vsc_ipynb_file__', __file__) if '__file__' in dir() else '.').resolve()
if REPO_ROOT.name == 'notebooks':
    REPO_ROOT = REPO_ROOT.parent
elif not (REPO_ROOT / 'src').exists():
    REPO_ROOT = Path('.').resolve()
    if not (REPO_ROOT / 'src').exists():
        REPO_ROOT = REPO_ROOT.parent  # one level up from notebooks/

sys.path.insert(0, str(REPO_ROOT / 'src'))

# ── Directory constants ────────────────────────────────────────────────────────
WORLDS_DIR    = REPO_ROOT / 'worlds'
REQUESTS_DIR  = REPO_ROOT / 'data' / 'user_requests'
EPISODES_DIR  = REPO_ROOT / 'outputs' / 'episodes'
EVAL_DIR      = REPO_ROOT / 'outputs' / 'eval_results'

# ── Python executable (use the same interpreter as this notebook) ─────────────
PYTHON = sys.executable

print(f'REPO_ROOT : {REPO_ROOT}')
print(f'PYTHON    : {PYTHON}')

---
## Stage 1 · World Generation

In [ ]:
# ── [Config] Stage 1 ──────────────────────────────────────────────────────────
S1_SET_NAME   = 'batch_001'   # sub-folder under worlds/
S1_N_WORLDS   = 3             # number of worlds to generate
S1_BASE_SEED  = 42            # world i gets seed BASE_SEED + i

# World generator overrides (None = use script default)
# Default: num_cities_per_region=4 (min for full complexity range)
S1_NUM_CITIES_PER_REGION    = None   # override e.g. 6 for richer worlds
S1_NUM_REGIONS               = None
S1_NUM_DISTRICTS_PER_CITY    = None
S1_NUM_HOTELS_PER_DISTRICT   = None
S1_NUM_EVENTS_PER_CITY       = None

In [ ]:
# ── [Launch] Stage 1 ──────────────────────────────────────────────────────────
_s1_cmd = [
    PYTHON, str(REPO_ROOT / 'scripts' / 'generate_worlds.py'),
    '--set_name', S1_SET_NAME,
    '--n_worlds', str(S1_N_WORLDS),
    '--base_seed', str(S1_BASE_SEED),
]
for flag, val in [
    ('--num_cities_per_region',  S1_NUM_CITIES_PER_REGION),
    ('--num_regions',            S1_NUM_REGIONS),
    ('--num_districts_per_city', S1_NUM_DISTRICTS_PER_CITY),
    ('--num_hotels_per_district',S1_NUM_HOTELS_PER_DISTRICT),
    ('--num_events_per_city',    S1_NUM_EVENTS_PER_CITY),
]:
    if val is not None:
        _s1_cmd += [flag, str(val)]

_s1_log = REPO_ROOT / 'outputs' / '.nb_logs' / f's1_{datetime.now(timezone.utc).strftime("%H%M%S")}.log'
_s1_log.parent.mkdir(parents=True, exist_ok=True)
_s1_fh  = open(_s1_log, 'w', encoding='utf-8', buffering=1)
_s1_env = {**os.environ, 'PYTHONUTF8': '1', 'PYTHONIOENCODING': 'utf-8', 'PYTHONUNBUFFERED': '1'}
_s1_proc = subprocess.Popen(_s1_cmd, stdout=_s1_fh, stderr=_s1_fh, cwd=str(REPO_ROOT), env=_s1_env)

print(f'[Stage 1] PID {_s1_proc.pid} started')
print(f'Log: {_s1_log}')
print(f'CMD: {" ".join(str(c) for c in _s1_cmd)}')

In [ ]:
# ── [Progress] Stage 1 — re-run this cell to check ───────────────────────────
_s1_world_dir = WORLDS_DIR / S1_SET_NAME
_s1_done = list(_s1_world_dir.glob('world_*')) if _s1_world_dir.exists() else []
_s1_rc   = _s1_proc.poll() if '_s1_proc' in dir() else None

print(f'Worlds created : {len(_s1_done)} / {S1_N_WORLDS}')
for w in sorted(_s1_done):
    print(f'  {w.name}')
if _s1_rc is None:
    print('Process: still running')
elif _s1_rc == 0:
    print('Process: finished OK')
else:
    print(f'Process: exit code {_s1_rc} — check log:')
    print(_s1_log.read_text(encoding='utf-8')[-2000:])

---
## Stage 2 · Request Generation

In [ ]:
# ── [Config] Stage 2 ──────────────────────────────────────────────────────────
S2_WORLD_SET      = S1_SET_NAME   # which world set to draw from
S2_N_TOTAL        = 30            # total requests across all worlds
S2_COLLECTION     = ''            # sub-folder name; '' = auto UTC timestamp
S2_SEED           = 42

# Complexity mix — relative weights (need not sum to 100; 0 excludes a tier).
# Set all to 1 for uniform distribution (no rejection sampling overhead).
S2_WEIGHT_LOW     = 1
S2_WEIGHT_MED     = 1
S2_WEIGHT_HIGH    = 1

In [ ]:
# ── [Launch] Stage 2 ──────────────────────────────────────────────────────────
_s2_cmd = [
    PYTHON, str(REPO_ROOT / 'scripts' / 'generate_eval_dataset.py'),
    '--world_set', str(WORLDS_DIR / S2_WORLD_SET),
    '--n_total', str(S2_N_TOTAL),
    '--seed', str(S2_SEED),
    '--output_dir', str(REQUESTS_DIR),
    '--complexity_weights', str(S2_WEIGHT_LOW), str(S2_WEIGHT_MED), str(S2_WEIGHT_HIGH),
]
if S2_COLLECTION.strip():
    _s2_cmd += ['--collection', S2_COLLECTION.strip()]

_s2_ts  = datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S')
_s2_log = REPO_ROOT / 'outputs' / '.nb_logs' / f's2_{_s2_ts}.log'
_s2_log.parent.mkdir(parents=True, exist_ok=True)
_s2_fh  = open(_s2_log, 'w', encoding='utf-8', buffering=1)
_s2_env = {**os.environ, 'PYTHONUTF8': '1', 'PYTHONIOENCODING': 'utf-8', 'PYTHONUNBUFFERED': '1'}
_s2_start_ts = time.time()
_s2_proc = subprocess.Popen(_s2_cmd, stdout=_s2_fh, stderr=_s2_fh, cwd=str(REPO_ROOT), env=_s2_env)

print(f'[Stage 2] PID {_s2_proc.pid} started')
print(f'Log: {_s2_log}')
print(f'CMD: {" ".join(str(c) for c in _s2_cmd)}')

In [ ]:
# ── [Progress] Stage 2 — re-run this cell to check ───────────────────────────
_s2_rc = _s2_proc.poll() if '_s2_proc' in dir() else None

# When S2_COLLECTION is explicitly set, count files in that folder directly.
# Mtime filtering would exclude pre-existing files when appending to an existing collection.
if S2_COLLECTION.strip():
    _s2_coll_dir = REQUESTS_DIR / S2_COLLECTION.strip()
    _s2_new = list(_s2_coll_dir.glob('request_*.json')) if _s2_coll_dir.exists() else []
    print(f'Collection      : {S2_COLLECTION.strip()}')
else:
    # Auto collection — count only what this launch produced via mtime
    _s2_new = [
        f for f in REQUESTS_DIR.rglob('request_*.json')
        if '_s2_start_ts' in dir() and f.stat().st_mtime >= _s2_start_ts
    ] if REQUESTS_DIR.exists() else []
    # Resolve the auto-generated collection name from the log
    if '_s2_log' in dir() and _s2_log.exists():
        for _line in _s2_log.read_text(encoding='utf-8').splitlines():
            if 'collection' in _line.lower() or 'output dir' in _line.lower():
                print(f'Collection      : {_line.strip()}')
                break

print(f'Requests saved  : {len(_s2_new)} / {S2_N_TOTAL}')
if _s2_rc is None:
    print('Process: still running')
elif _s2_rc == 0:
    print('Process: finished OK')
else:
    print(f'Process: exit code {_s2_rc} — log tail:')
    print(_s2_log.read_text(encoding='utf-8')[-2000:] if '_s2_log' in dir() and _s2_log.exists() else '(no log)')

---
## Stage 3 · Agent Run

In [ ]:
# ── [Config] Stage 3 ──────────────────────────────────────────────────────────
# agent_mode: 'raw' | 'llm_summary' | 'compressor' | 'mcts_compressor'
S3_AGENT_MODE         = 'raw'
S3_COLLECTION         = ''            # collection name from Stage 2; '' = pick from list below
S3_LLM_MODEL_ID       = 'openai/gpt-4o-mini'
S3_MAX_STEPS          = 30
S3_COMPRESS_EVERY_N   = 5
S3_AUGMENTATION_ID    = None          # AugmentationRegistry ID; None for raw/llm_summary
S3_PROMPT_ID          = None          # PromptRegistry ID; None = default v2 prompt
S3_SEED               = 42

# If S3_COLLECTION is blank, pick the most recent collection automatically:
if not S3_COLLECTION.strip() and REQUESTS_DIR.exists():
    _colls = sorted(
        [d for d in REQUESTS_DIR.iterdir() if d.is_dir() and any(d.glob('request_*.json'))],
        key=lambda d: d.stat().st_mtime, reverse=True,
    )
    if _colls:
        S3_COLLECTION = _colls[0].name
        print(f'Auto-selected collection: {S3_COLLECTION}')
    else:
        print('WARNING: no request collections found — run Stage 2 first')
else:
    print(f'Collection: {S3_COLLECTION}')

In [ ]:
# ── [Launch] Stage 3 ──────────────────────────────────────────────────────────
_s3_cmd = [
    PYTHON, str(REPO_ROOT / 'scripts' / 'run_agent_batch.py'),
    '--collection', S3_COLLECTION,
    '--agent_mode', S3_AGENT_MODE,
    '--llm_model_id', S3_LLM_MODEL_ID,
    '--max_steps', str(S3_MAX_STEPS),
    '--compress_every_n_steps', str(S3_COMPRESS_EVERY_N),
    '--seed', str(S3_SEED),
    '--worlds_root', str(WORLDS_DIR),
    '--requests_dir', str(REQUESTS_DIR),
    '--output_dir', str(EPISODES_DIR),
]
if S3_AUGMENTATION_ID:
    _s3_cmd += ['--augmentation_id', S3_AUGMENTATION_ID]
if S3_PROMPT_ID:
    _s3_cmd += ['--prompt_id', S3_PROMPT_ID]

_s3_ts  = datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S')
_s3_log = REPO_ROOT / 'outputs' / '.nb_logs' / f's3_{_s3_ts}.log'
_s3_log.parent.mkdir(parents=True, exist_ok=True)
_s3_fh  = open(_s3_log, 'w', encoding='utf-8', buffering=1)
_s3_env = {**os.environ, 'PYTHONUTF8': '1', 'PYTHONIOENCODING': 'utf-8', 'PYTHONUNBUFFERED': '1'}
_s3_start_ts = time.time()
_s3_n_requests = len(list((REQUESTS_DIR / S3_COLLECTION).glob('request_*.json')))
_s3_proc = subprocess.Popen(_s3_cmd, stdout=_s3_fh, stderr=_s3_fh, cwd=str(REPO_ROOT), env=_s3_env)

print(f'[Stage 3] PID {_s3_proc.pid} started')
print(f'Requests to run : {_s3_n_requests}')
print(f'Log: {_s3_log}')

In [ ]:
# ── [Progress] Stage 3 — re-run this cell to check ───────────────────────────
_s3_rc = _s3_proc.poll() if '_s3_proc' in dir() else None
_s3_new_eps = [
    f for f in EPISODES_DIR.glob('ep_*.json')
    if '_s3_start_ts' in dir() and f.stat().st_mtime >= _s3_start_ts
] if EPISODES_DIR.exists() else []

_s3_total = _s3_n_requests if '_s3_n_requests' in dir() else '?'
print(f'Episodes saved : {len(_s3_new_eps)} / {_s3_total}')

# Parse EPISODE_DONE lines from log for a finer-grained count
if '_s3_log' in dir() and _s3_log.exists():
    _done_lines = [l for l in _s3_log.read_text(encoding='utf-8').splitlines() if l.startswith('EPISODE_DONE')]
    if _done_lines:
        print(f'Agent reported : {_done_lines[-1]}')

if _s3_rc is None:
    print('Process: still running')
elif _s3_rc == 0:
    print('Process: finished OK')
else:
    print(f'Process: exit code {_s3_rc} — log tail:')
    print(_s3_log.read_text(encoding='utf-8')[-2000:] if '_s3_log' in dir() and _s3_log.exists() else '(no log)')

---
## Stage 4 · Scoring

In [ ]:
# ── [Config] Stage 4 ──────────────────────────────────────────────────────────
# eval_mode: 'deterministic' | 'llm_judge' | 'full'
# 'deterministic' is free (no API calls); 'full' adds LLM judge scoring.
S4_EVAL_MODE       = 'deterministic'
S4_RUN_NAME        = 'sweep_D-raw-baseline'   # groups related runs in the run table
S4_AUGMENTATION_ID = S3_AUGMENTATION_ID       # inherit from Stage 3
S4_PROMPT_ID       = S3_PROMPT_ID             # inherit from Stage 3
S4_JUDGE_MODEL     = 'openai/gpt-4o-mini'     # only used when eval_mode != deterministic
S4_AGENT_MODE_FILTER = ''                     # '' = score all agent modes
S4_PARENT_RUN_ID   = None                     # set when re-scoring an existing run
S4_NOTES           = ''                       # free-text

# Episode selection — pick ONE:
# Option A: score only episodes from this Stage 3 run (default)
# Option B: score ALL episodes in EPISODES_DIR (set S4_SCORE_ALL = True)
# Option C: score specific episode IDs (set S4_EPISODE_IDS = ['uuid1', 'uuid2'])
S4_SCORE_ALL      = False
S4_EPISODE_IDS    = []   # explicit list; empty = use Option A or B

In [ ]:
# ── [Launch] Stage 4 ──────────────────────────────────────────────────────────
_s4_cmd = [
    PYTHON, str(REPO_ROOT / 'scripts' / 'run_eval.py'),
    '--eval_mode', S4_EVAL_MODE,
    '--episodes_dir', str(EPISODES_DIR),
    '--eval_dir', str(EVAL_DIR),
]

# Episode selection
if S4_EPISODE_IDS:
    _s4_cmd += ['--episode_ids'] + S4_EPISODE_IDS
elif not S4_SCORE_ALL and '_s3_start_ts' in dir():
    # Score only new episodes from Stage 3
    _new_ep_ids = [
        f.stem.removeprefix('ep_')
        for f in EPISODES_DIR.glob('ep_*.json')
        if f.stat().st_mtime >= _s3_start_ts
    ]
    if _new_ep_ids:
        _s4_cmd += ['--episode_ids'] + _new_ep_ids
        print(f'Scoring {len(_new_ep_ids)} new episodes from Stage 3')
    else:
        _s4_cmd += ['--all']
        print('No new Stage-3 episodes found — scoring all episodes')
else:
    _s4_cmd += ['--all']
    print('Scoring ALL episodes in EPISODES_DIR')

if S4_RUN_NAME:            _s4_cmd += ['--run_name', S4_RUN_NAME]
if S4_AUGMENTATION_ID:     _s4_cmd += ['--augmentation_id', S4_AUGMENTATION_ID]
if S4_PROMPT_ID:           _s4_cmd += ['--prompt_id', S4_PROMPT_ID]
if S4_AGENT_MODE_FILTER:   _s4_cmd += ['--agent_mode', S4_AGENT_MODE_FILTER]
if S4_PARENT_RUN_ID:       _s4_cmd += ['--parent_run_id', S4_PARENT_RUN_ID]
if S4_NOTES:               _s4_cmd += ['--note', S4_NOTES]
if S4_EVAL_MODE != 'deterministic':
    _s4_cmd += ['--judge_model', S4_JUDGE_MODEL]

_s4_ts  = datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S')
_s4_log = REPO_ROOT / 'outputs' / '.nb_logs' / f's4_{_s4_ts}.log'
_s4_log.parent.mkdir(parents=True, exist_ok=True)
_s4_fh  = open(_s4_log, 'w', encoding='utf-8', buffering=1)
_s4_env = {**os.environ, 'PYTHONUTF8': '1', 'PYTHONIOENCODING': 'utf-8', 'PYTHONUNBUFFERED': '1'}
_s4_start_ts = time.time()
_s4_proc = subprocess.Popen(_s4_cmd, stdout=_s4_fh, stderr=_s4_fh, cwd=str(REPO_ROOT), env=_s4_env)

print(f'[Stage 4] PID {_s4_proc.pid} started')
print(f'Log: {_s4_log}')
print(f'CMD: {" ".join(str(c) for c in _s4_cmd)}')

In [ ]:
# ── [Progress] Stage 4 — re-run this cell to check ───────────────────────────
_s4_rc = _s4_proc.poll() if '_s4_proc' in dir() else None

# Count results.jsonl lines written after launch
_s4_scored = 0
for _rj in EVAL_DIR.rglob('results.jsonl'):
    if '_s4_start_ts' in dir() and _rj.stat().st_mtime >= _s4_start_ts:
        _s4_scored += sum(1 for _ in _rj.read_text(encoding='utf-8').strip().splitlines() if _)

print(f'Results scored : {_s4_scored}')

# Show live manifest progress if the run_id is known
if '_s4_log' in dir() and _s4_log.exists():
    for _line in _s4_log.read_text(encoding='utf-8').splitlines():
        if 'run_id' in _line and 'manifest' in _line.lower():
            print(' ', _line.strip())

if _s4_rc is None:
    print('Process: still running')
elif _s4_rc == 0:
    print('Process: finished OK')
else:
    print(f'Process: exit code {_s4_rc} — log tail:')
    print(_s4_log.read_text(encoding='utf-8')[-3000:] if '_s4_log' in dir() and _s4_log.exists() else '(no log)')

---
## Analysis

The cells below are independent of the pipeline stages above — they read from disk and can be run at any time.

### A · Run Inventory

Quick overview of all completed eval runs — use this to find run IDs to filter on below.

In [ ]:
from optimized_llm_planning_memory.evaluation.results_loader import (
    load_flat_results,
    list_runs_summary,
    make_comparison_table,
    filter_results,
)

runs = list_runs_summary(EVAL_DIR)
print(f'{len(runs)} eval run(s) found')
runs

### B · Load Flat Results

Each row = one evaluated episode. Multi-level columns: `ids`, `run`, `config`, `episode`, `request`, `metrics`, `det`, `llm`.

In [ ]:
# ── [Config] ──────────────────────────────────────────────────────────────────
# Filter to specific runs (None = load all)
LOAD_RUN_IDS            = None   # e.g. ['20260506_120000_abc12345']
# Keep only the most recent result per (request, agent_mode, metric_version) triple
LOAD_LATEST_PER_KEY     = True

In [ ]:
df = load_flat_results(
    eval_dir=EVAL_DIR,
    episodes_dir=EPISODES_DIR,
    requests_dir=REQUESTS_DIR,
    run_ids=LOAD_RUN_IDS,
    latest_per_eval_key=LOAD_LATEST_PER_KEY,
)

print(f'Loaded {len(df)} result rows')
print(f'Column groups : {df.columns.get_level_values(0).unique().tolist()}')
print(f'Det metrics   : {df["det"].columns.tolist() if "det" in df.columns.get_level_values(0) else "(none)"}')
print(f'LLM metrics   : {df["llm"].columns.tolist() if "llm" in df.columns.get_level_values(0) else "(none — deterministic-only run)"}')
df.head(3)

### C · Metric Coverage Check

Which episodes have LLM judge scores? Use this to identify runs that need re-scoring with `--eval_mode llm_judge`.

In [ ]:
_coverage = df[[('config', 'agent_mode'), ('config', 'run_name'), ('metrics', 'has_llm_scores')]].copy()
_coverage.columns = ['agent_mode', 'run_name', 'has_llm_scores']
_coverage.groupby(['run_name', 'agent_mode', 'has_llm_scores']).size().rename('n_episodes').reset_index()

### D · Filtering

Subset the flat DataFrame before computing tables or charts.

In [ ]:
# ── [Config] ──────────────────────────────────────────────────────────────────
FILTER_AGENT_MODES      = None   # e.g. ['raw', 'llm_summary', 'compressor']
FILTER_AUGMENTATION_IDS = None   # e.g. ['tgad-trained-001']
FILTER_PROMPT_IDS       = None
FILTER_RUN_IDS          = None
FILTER_COMPLEXITY_TIERS = None   # e.g. ['medium', 'high']
FILTER_METRIC_VERSION   = None   # e.g. 'v1'
FILTER_HAS_LLM_SCORES   = None   # True = only rows with LLM judge scores
FILTER_LATEST_ONLY      = True   # deduplicate re-runs by eval_key

In [ ]:
df_filtered = filter_results(
    df,
    agent_modes=FILTER_AGENT_MODES,
    augmentation_ids=FILTER_AUGMENTATION_IDS,
    prompt_ids=FILTER_PROMPT_IDS,
    run_ids=FILTER_RUN_IDS,
    complexity_tiers=FILTER_COMPLEXITY_TIERS,
    metric_version=FILTER_METRIC_VERSION,
    has_llm_scores=FILTER_HAS_LLM_SCORES,
    latest_per_eval_key=FILTER_LATEST_ONLY,
)
print(f'{len(df_filtered)} rows after filtering')
df_filtered[[('config', 'agent_mode'), ('config', 'augmentation_id'),
             ('request', 'complexity_tier'), ('metrics', 'overall_score'),
             ('det', 'hard_constraint_ratio')]].head(10)

### E · Comparison / Ablation Table

Pivot `df_filtered` into a mean ± std table, grouped by one or more config dimensions.  
This is the primary output for paper tables and presentations.

In [ ]:
# ── [Config] ──────────────────────────────────────────────────────────────────
# group_by: any column name(s) from the 'config' group
TABLE_GROUP_BY      = 'agent_mode'          # or ['agent_mode', 'augmentation_id']
TABLE_METRIC_GROUP  = 'det'                 # 'det' or 'llm'
TABLE_METRICS       = [                     # None = all metrics in the group
    'hard_constraint_ratio',
    'soft_constraint_score',
    'budget_adherence',
    'tool_efficiency',
    'completion_rate',
]
TABLE_STAT          = 'mean±std'            # 'mean±std' | 'mean' | 'full'

In [ ]:
tbl = make_comparison_table(
    df_filtered,
    group_by=TABLE_GROUP_BY,
    metrics=TABLE_METRICS,
    metric_group=TABLE_METRIC_GROUP,
    include_overall=True,
    stat=TABLE_STAT,
)
print(f'Conditions: {list(tbl.index)}')
tbl

### F · Complexity Stratification

Does performance differ by request complexity tier?  
Cross-tab: condition × complexity → mean metrics.

In [ ]:
STRAT_METRIC   = 'hard_constraint_ratio'   # pick any det metric
STRAT_GROUP_BY = 'agent_mode'              # condition to compare

_strat_df = df_filtered[[
    ('config', STRAT_GROUP_BY),
    ('request', 'complexity_tier'),
    ('det', STRAT_METRIC),
]].copy()
_strat_df.columns = [STRAT_GROUP_BY, 'complexity_tier', STRAT_METRIC]

_strat_df.groupby([STRAT_GROUP_BY, 'complexity_tier'])[STRAT_METRIC].agg(['mean', 'std', 'count']).round(3)

### G · Export

In [ ]:
EXPORT_PATH = REPO_ROOT / 'outputs' / 'eval_flat_results.csv'

# Flatten multi-level columns to 'group.field' strings for CSV compatibility
_df_export = df_filtered.copy()
_df_export.columns = [f'{g}.{f}' for g, f in _df_export.columns]
_df_export.to_csv(EXPORT_PATH, index=False)
print(f'Saved {len(_df_export)} rows → {EXPORT_PATH}')